In [1]:
from pipeline import create_pipeline, save_results_from_cache


## 1. 创建 Pipeline


In [2]:
import os

# 使用便捷函数创建（会设置全局配置）；密钥请用环境变量，勿写入仓库
# pipeline = create_pipeline(
#     api_key=os.environ["ARK_API_KEY"],
#     model="<your-endpoint-model-id>",
#     base_url="https://ark.cn-beijing.volces.com/api/v3"
# )
# pipeline = create_pipeline(
#     api_key=os.environ["OPENROUTER_API_KEY"],
#     model="openai/gpt-5",
#     base_url="https://openrouter.ai/api/v1"
# )

# google/gemini-2.5-pro
# openai/gpt-5
# deepseek/deepseek-v3.1-terminus
# anthropic/claude-sonnet-4.5


pipeline = create_pipeline(
    api_key=os.environ["DASHSCOPE_API_KEY"],
    model="qwen3-max",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)


## 3. 测试数据


In [3]:
# 聚类功能已增强：会根据provenances.json中的type字段分类聚类
# 生成三个文件：
#   1. gen/cluster_guideline.json - 指南类数据聚类
#   2. gen/cluster_expert.json - 专家共识类数据聚类
#   3. gen/cluster.json - 混合数据聚类（所有数据）
clusters, bucket_index = pipeline.cluster_provenances(save_to_file=True,load_filepath='gen/provenances.json',filepath='gen/cluster.json')

[load_provenances] 从 gen/provenances.json 加载了 191 条推荐意见
[第二阶段] 数据分类: guideline=87, expert=104, 总计=191
[第二阶段-guideline] 开始对 87 条guideline推荐意见进行LSH聚类...
[第二阶段-guideline] 聚类完成：87 条推荐 -> 16 个聚类
[save_clusters] 保存了 16 个聚类，包含 87 条推荐意见到 gen\cluster_guideline.json
[第二阶段-guideline] 已保存到 gen\cluster_guideline.json
[第二阶段-expert] 开始对 104 条expert推荐意见进行LSH聚类...
[第二阶段-expert] 聚类完成：104 条推荐 -> 19 个聚类
[save_clusters] 保存了 19 个聚类，包含 104 条推荐意见到 gen\cluster_expert.json
[第二阶段-expert] 已保存到 gen\cluster_expert.json
[第二阶段-混合] 开始对 191 条推荐意见进行LSH聚类...
[第二阶段-混合] 聚类完成：191 条推荐 -> 36 个聚类
[save_clusters] 保存了 36 个聚类，包含 191 条推荐意见到 gen/cluster.json
[第二阶段-混合] 已保存到 gen/cluster.json


In [5]:
async def main():
    res = await pipeline.process_clusters_async(
        load_filepath='gen/cluster.json',
        load_from_file=True,
        cluster_cache_path='gen/cluster_cache.json',
        persist_cluster_cache=True,
        verbose=True         # 打印进度
    )
    print("done", {k: len(v) for k, v in res.items()})

await main()

[load_clusters] 从 gen/cluster.json 加载了 45 个聚类
[StandardLibrary] 已加载: 183 terms, 251 meds, 495 predicates
[第三阶段-async] cluster 15 完成 (1/45)
[第三阶段-async] cluster 12 完成 (2/45)
[第三阶段-async] cluster 8 完成 (3/45)
[第三阶段-async] cluster 16 完成 (4/45)
[第三阶段-async] cluster 29 完成 (5/45)
[第三阶段-async] cluster 20 完成 (6/45)
[第三阶段-async] cluster 33 完成 (7/45)
[第三阶段-async] cluster 31 完成 (8/45)
[第三阶段-async] cluster 1 完成 (9/45)
[第三阶段-async] cluster 27 完成 (10/45)
[第三阶段-async] cluster 10 完成 (11/45)
[第三阶段-async] cluster 32 完成 (12/45)
[第三阶段-async] cluster 11 完成 (13/45)
[第三阶段-async] cluster 9 完成 (14/45)
[第三阶段-async] cluster 38 完成 (15/45)
[第三阶段-async] cluster 17 完成 (16/45)
[第三阶段-async] cluster 0 完成 (17/45)
[第三阶段-async] cluster 26 完成 (18/45)
[第三阶段-async] cluster 19 完成 (19/45)
[第三阶段-async] cluster 30 完成 (20/45)
[第三阶段-async] cluster 28 完成 (21/45)
[第三阶段-async] cluster 13 完成 (22/45)
[第三阶段-async] cluster 2 完成 (23/45)
[第三阶段-async] cluster 35 完成 (24/45)
[第三阶段-async] cluster 36 完成 (25/45)
[第三阶段-async] cluster 37 完成 (26/45)

In [3]:
res = pipeline.extract_terms_stage(load_filepath='gen/cluster.json',
        load_from_file=True,
        cluster_cache_path='gen/cluster_cache.json',
        persist_cluster_cache=True,
        max_concurrency=10,  # 并发度

        )


[load_clusters] 从 gen/cluster.json 加载了 36 个聚类
[StandardLibrary] 已加载: 183 terms, 251 meds, 495 predicates
[process_cluster_terms][cluster 6] terms=2 med_terms=4
[process_cluster_terms][cluster 8] terms=3 med_terms=5
[process_cluster_terms][cluster 7] terms=2 med_terms=5
[process_cluster_terms][cluster 5] terms=4 med_terms=1
[process_cluster_terms][cluster 9] terms=5 med_terms=2
[process_cluster_terms][cluster 0] terms=5 med_terms=5
[process_cluster_terms][cluster 2] terms=9 med_terms=3
[process_cluster_terms][cluster 3] terms=7 med_terms=7
[process_cluster_terms][cluster 4] terms=8 med_terms=4
[process_cluster_terms][cluster 1] terms=9 med_terms=7
[process_cluster_terms][cluster 10] terms=2 med_terms=7
[process_cluster_terms][cluster 14] terms=3 med_terms=4
[process_cluster_terms][cluster 16] terms=2 med_terms=3
[process_cluster_terms][cluster 13] terms=10 med_terms=3
[process_cluster_terms][cluster 19] terms=5 med_terms=1
[process_cluster_terms][cluster 17] terms=5 med_terms=3
[process

In [4]:
res = pipeline.extract_predicates_stage(load_filepath='gen/cluster.json',
        load_from_file=True,
        cluster_cache_path='gen/cluster_cache.json',
        persist_cluster_cache=True,
        max_concurrency=10,  # 并发度

        )


[load_clusters] 从 gen/cluster.json 加载了 36 个聚类
[load_cluster_cache] loaded 36 clusters from gen/cluster_cache.json
[process_cluster_predicates][cluster 3] preds=5 (复用缓存: terms=True, meds=True)
[process_cluster_predicates][cluster 1] preds=9 (复用缓存: terms=True, meds=True)
[process_cluster_predicates][cluster 6] preds=7 (复用缓存: terms=True, meds=True)
[process_cluster_predicates][cluster 5] preds=5 (复用缓存: terms=True, meds=True)
[process_cluster_predicates][cluster 8] preds=10 (复用缓存: terms=True, meds=True)
[process_cluster_predicates][cluster 0] preds=9 (复用缓存: terms=True, meds=True)
[process_cluster_predicates][cluster 9] preds=5 (复用缓存: terms=True, meds=True)
[process_cluster_predicates][cluster 7] preds=15 (复用缓存: terms=True, meds=True)
[process_cluster_predicates][cluster 2] preds=9 (复用缓存: terms=True, meds=True)
[process_cluster_predicates][cluster 11] preds=7 (复用缓存: terms=True, meds=True)
[process_cluster_predicates][cluster 10] preds=6 (复用缓存: terms=True, meds=True)
[process_cluster_predica

In [5]:
res = pipeline.extract_rules_stage(load_filepath='gen/cluster.json',
        load_from_file=True,
        cluster_cache_path='gen/cluster_cache.json',
        persist_cluster_cache=True,
        max_concurrency=10,  # 并发度

        )


[load_clusters] 从 gen/cluster.json 加载了 36 个聚类
[load_cluster_cache] loaded 36 clusters from gen/cluster_cache.json
[process_cluster_rules][cluster 5] rules=2 (复用缓存: terms=True, meds=True, preds=True)
[process_cluster_rules][cluster 9] rules=3 (复用缓存: terms=True, meds=True, preds=True)
[process_cluster_rules][cluster 0] rules=4 (复用缓存: terms=True, meds=True, preds=True)
[process_cluster_rules][cluster 1] rules=4 (复用缓存: terms=True, meds=True, preds=True)
[process_cluster_rules][cluster 8] rules=4 (复用缓存: terms=True, meds=True, preds=True)
[process_cluster_rules][cluster 6] rules=4 (复用缓存: terms=True, meds=True, preds=True)
[process_cluster_rules][cluster 7] rules=4 (复用缓存: terms=True, meds=True, preds=True)
[process_cluster_rules][cluster 4] rules=7 (复用缓存: terms=True, meds=True, preds=True)
[process_cluster_rules][cluster 2] rules=8 (复用缓存: terms=True, meds=True, preds=True)
[process_cluster_rules][cluster 3] rules=8 (复用缓存: terms=True, meds=True, preds=True)
[process_cluster_rules][cluster 13] 

In [3]:
# Save aggregated results from cluster cache to gen/
save_results_from_cache(cluster_cache_path='gen/cluster_cache.json', gen_dir='gen', save_provenances=False)


[load_cluster_cache] loaded 36 clusters from gen/cluster_cache.json
[save_to_gen] terms.json: +56 -> total 56
[save_to_gen] med_terms.json: +39 -> total 39
[save_to_gen] predicates.json: +200 -> total 200
[save_to_gen] rules.json: +338 -> total 338
✅ 已创建 gen\cluster_final.json，包含 36 个 cluster


In [1]:
import pandas as pd

In [3]:
gold = pd.read_excel("trans.xlsx")

In [4]:
gold

,id,label,condition,action_subjects,action_permission,action_requirements,provenance_quote,provenance_recommendation_grade,provenance_evidence_level,provenance_source
0,rule.dkd_metformin_sglt2i_intolerant_glp1ra_re...,DKD患者不能耐受二甲双胍或SGLT2i时推荐GLP-1 RA,pred.Has.cond.dkd AND ((pred.On.med.metformin ...,med.class.glp1ra,recommend,NaN,对于DKD患者使用二甲双胍联合SGLT2i血糖仍未达标或不能耐受上述药物时，推荐使用具有延缓...,1,B,中国糖尿病肾脏病基层管理指南
1,rule.overweight_obesity_diabetes_hypertension_...,推荐超重或肥胖的糖尿病合并高血压患者使用二甲双胍,pred.Has.obs.obesity,med.metformin,recommend,NaN,超重或肥胖也是糖尿病合并高血压的重要危险因素，除了生活方式干预外，药物治疗也应考虑体重管理，...,None,None,糖尿病合并高血压患者管理指南
2,rule.overweight_obesity_diabetes_hypertension_...,推荐超重或肥胖的糖尿病合并高血压患者使用SGLT2抑制剂,pred.Has.obs.obesity,med.class.sglt2,recommend,NaN,超重或肥胖也是糖尿病合并高血压的重要危险因素，除了生活方式干预外，药物治疗也应考虑体重管理，...,None,None,糖尿病合并高血压患者管理指南
3,rule.overweight_obesity_diabetes_hypertension_...,推荐超重或肥胖的糖尿病合并高血压患者使用GLP-1受体激动剂,pred.Has.obs.obesity,med.class.glp1,recommend,NaN,超重或肥胖也是糖尿病合并高血压的重要危险因素，除了生活方式干预外，药物治疗也应考虑体重管理，...,None,None,糖尿病合并高血压患者管理指南
4,rule.prolonged_fasting_surgery_sglt2i_stop,长时间禁食或手术时停用SGLT2抑制剂,pred.Has.proc.surgery,med.class.sglt2,stop,NaN,在长时间禁食、手术或严重疾病（可能有更大的酮症风险）时，停用SGLT2i是合理的,None,None,中国成人IgA肾病及IgA血管炎肾炎临床实践指南（2025）
...,...,...,...,...,...,...,...,...,...,...
401,rule.hypertension_diabetes_sglt2i_recommend,高血压合并糖尿病患者推荐使用SGLT2i,pred.Has.cond.hypertension AND pred.Has.cond.t2dm,med.class.sglt2,recommend,NaN,"推荐高血压合并糖尿病患者使用SGLT2i和GLP-1RA以降低心肾事件风险,同时具有一定降压作用",Ⅰ,A,中国高血压防治指南(2024)
402,rule.hypertension_diabetes_glp1ra_recommend,高血压合并糖尿病患者推荐使用GLP-1RA,pred.Has.cond.hypertension AND pred.Has.cond.t2dm,med.class.glp1ra,recommend,NaN,"推荐高血压合并糖尿病患者使用SGLT2i和GLP-1RA以降低心肾事件风险,同时具有一定降压作用",Ⅰ,A,中国高血压防治指南(2024)
403,rule.ckd_t2dm_elderly_prefer_sglt2i,老年CKD合并T2DM患者在ACEI/ARB基础上优选SGLT2i,pred.Value.demographics.age.ge.65 AND pred.Has...,med.class.sglt2,recommend,NaN,老年 CKD 合并 2 型糖尿病（type 2 diabetes mellitus，T2DM...,建议,A,老年慢性肾脏病综合管理指南(2025年版)
404,rule.t2dm_ckd_elderly_rasi_sglt2i_finerenone_c...,老年T2DM合并CKD患者推荐RASi、SGLT2i、非奈利酮联合治疗,pred.Value.demographics.age.ge.65 AND pred.Has...,"med.class.rasi, med.class.sglt2, med.finerenone",recommend,"定期监测血压、血钾及肾功能, 加用SGLT2i和/或nsMRA时建议间隔3-4周, 起始治疗...",推荐 RASi、SGLT2i、非奈利酮三类药物（“肾三联”）遵照合理、序贯、个体化原则，联合...,None,None,老年慢性肾脏病综合管理指南(2025年版)
